In [ ]:
# ========== 导入：多模态客服 Nova 需要的标准库与第三方包 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 re：用正则从用户消息里抓工单号（如 rt-1001）
import re
# 导入标准库 time：生成工单创建时间戳（strftime）
import time
# 导入标准库 json：本文件中已导入（预留/与周边代码一致），此处保持原样
import json
# 导入标准库 sqlite3：本地 SQLite 存工单（tickets 表）
import sqlite3
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免写进代码
from dotenv import load_dotenv
# 导入 gradio：快速搭 Web UI（ChatInterface + Audio）
import gradio as gr
# 从 openai 导入 OpenAI 客户端：Chat Completions + TTS（语音合成）
from openai import OpenAI


In [ ]:
# ========== 环境初始化：密钥 + OpenAI 客户端 + 数据库路径 ==========

# 加载 .env（默认不 override；参数保持原样）
load_dotenv()
# 创建 OpenAI 客户端；密钥从环境变量 OPENAI_API_KEY 读取
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# SQLite 文件路径：工单落盘到当前工作目录下的 nova_support.db
DB_PATH = "nova_support.db"


In [ ]:
# ========== 建表：若不存在则创建 tickets 工单表 ==========

def init_db():
    # 连接 SQLite（文件不存在会自动创建）
    conn = sqlite3.connect(DB_PATH)
    # 拿到游标（cursor），用来执行 SQL
    cur = conn.cursor()
    # CREATE TABLE IF NOT EXISTS：重复运行也安全，不会抹掉已有数据
    cur.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            ticket_id TEXT PRIMARY KEY,
            name TEXT,
            company TEXT,
            email TEXT,
            issue TEXT,
            priority TEXT,
            status TEXT,
            created_at TEXT
        )
    """)
    # 提交事务，把建表真正写进磁盘
    conn.commit()
    # 关闭连接，释放文件锁
    conn.close()


In [ ]:
# ========== 生成工单号：按现有行数递增，形如 RT-1001 ==========

def new_ticket_id():
    # 打开同一数据库，统计当前工单条数
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # COUNT(*)：表里已有多少行
    cur.execute("SELECT COUNT(*) FROM tickets")
    # fetchone()[0] 取出计数值
    count = cur.fetchone()[0]
    conn.close()
    # 编号规则：基数 1001 + 当前条数 → 第一条是 RT-1001
    return f"RT-{1001 + count}"


In [ ]:
# ========== 创建工单：写入一行 OPEN 状态的 ticket ==========

def create_ticket(name, company, email, issue, priority="P3"):
    # 先拿新工单号
    tid = new_ticket_id()
    # 本地时间字符串，写入 created_at
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # 参数化 INSERT（? 占位），避免 SQL 拼接注入；priority 统一大写；status 固定 OPEN
    cur.execute("""
        INSERT INTO tickets (ticket_id, name, company, email, issue, priority, status, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (tid, name, company, email, issue, priority.upper(), "OPEN", ts))
    conn.commit()
    conn.close()
    # 返回工单号与时间戳，给聊天回复拼接文案
    return tid, ts


In [ ]:
# ========== 查询工单：按 ticket_id 取一行，转成 dict ==========

def get_ticket(ticket_id):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # 参数化 WHERE，精确匹配工单号
    cur.execute("SELECT * FROM tickets WHERE ticket_id=?", (ticket_id,))
    row = cur.fetchone()
    conn.close()
    # 查无记录 → 返回 None，上层会提示 not found
    if not row:
        return None
    # 列名顺序必须与建表 / SELECT * 一致，zip 成字典方便按字段读
    keys = ["ticket_id", "name", "company", "email", "issue", "priority", "status", "created_at"]
    return dict(zip(keys, row))


In [ ]:
# ========== TTS：把回复文本合成语音文件（gpt-4o-mini-tts） ==========

def synthesize_speech(text):
    # 空字符串不合成，避免无效 API 调用
    if not text.strip():
        return None
    # 输出到系统临时目录下的固定文件名 nova_reply.mp3
    # 注意：本格依赖 Path / tempfile（需在运行环境可用；导入保持与原笔记本一致）
    output_path = Path(tempfile.gettempdir()) / "nova_reply.mp3"
    # with_streaming_response：边收边写文件；model / voice / input 字符串保持原样
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    ) as response:
        # 把音频流落盘到 output_path
        response.stream_to_file(output_path)
    # Gradio Audio 需要路径字符串
    return str(output_path)


In [ ]:
# ========== System Prompt：Nova 客服人设（影响模型行为的英文原文不翻译） ==========

# 三段引号字符串会原样发给 chat.completions；角色、能力列表、语气要求都写在这里
SYSTEM_PROMPT = """
You are Nova, the AI Support and Sales Assistant for Reallytics.ai.
You help customers with:
- Reporting issues (create tickets)
- Checking existing tickets
- Providing product/service information
- Explaining pricing ranges
- Reassuring integration compatibility with client systems
Respond in a professional, business tone.
"""


In [ ]:
# ========== 意图识别：用关键词/正则把消息路由到不同分支（非 LLM 分类） ==========

def detect_intent(message):
    # 统一小写，方便子串匹配
    text = message.lower()
    # 建单意图：出现 create/open/new ticket 或 issue/problem
    if any(k in text for k in ["create ticket", "open ticket", "new ticket", "issue", "problem"]):
        return "create_ticket"
    # 查单意图：消息里出现 rt-数字（工单号模式）
    if re.search(r"rt-\d+", text):
        return "check_ticket"
    # 报价意图
    if "price" in text or "cost" in text:
        return "pricing"
    # 集成兼容性意图
    if "integration" in text:
        return "integration"
    # 其余走通用 LLM 对话
    return "general"


In [ ]:
# ========== chat（中间版本）：只处理建单/查单并 yield 文本+语音后 return ==========
# 理念：Gradio 生成器协议 —— yield (text, audio_path)；后面单元格可能再定义更完整的 chat

def chat(message, history, model, name, company, email):
    # 把 Gradio messages 历史转成 OpenAI 风格的 role/content 列表（本版后面未用到 LLM）
    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    # 先做规则意图分类
    intent = detect_intent(message)

    # —— 建单分支 ——
    if intent == "create_ticket":
        # urgent/high → P2，否则默认 P3
        priority = "P2" if "urgent" in message.lower() or "high" in message.lower() else "P3"
        # 用表单里的 name/company/email + 整段 message 作为 issue 写入 DB
        tid, ts = create_ticket(name, company, email, message, priority)
        # 给用户看的英文回执（可运行字符串保持原样）
        text = f"A new support ticket has been created.\nTicket ID: {tid}\nCreated at: {ts}\nStatus: OPEN"
        # 同时返回 TTS 音频路径
        yield text, synthesize_speech(text)
        return

    # —— 查单分支 ——
    if intent == "check_ticket":
        # 从消息里抓第一个 rt-数字
        match = re.search(r"(rt-\d+)", message.lower())
        if match:
            # 工单号统一大写再查库
            ticket_id = match.group(1).upper()
            data = get_ticket(ticket_id)
            if data:
                text = (
                    f"Ticket {ticket_id} Details:\n"
                    f"Issue: {data['issue']}\n"
                    f"Status: {data['status']}\n"
                    f"Priority: {data['priority']}\n"
                    f"Created at: {data['created_at']}"
                )
            else:
                text = f"No ticket found with ID {ticket_id}."
        else:
            text = "Please provide a valid ticket ID."
        yield text, synthesize_speech(text)
        return


In [ ]:
# ========== chat（完整版）：建单/查单 + 通用流式 LLM + 结束后再 TTS ==========
# 说明：本格重新定义同名 chat，运行后会覆盖上一格；这是 Gradio 实际绑定的版本

def chat(message, history, model, name, company, email):
    # 空消息直接提示，不调 API
    if not message.strip():
        yield "Please type a message to start.", None
        return

    # Gradio history → OpenAI messages
    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    # 规则意图
    intent = detect_intent(message)
    # 预先占位：文本回复与音频路径
    reply, audio_path = "", None

    # —— 建单：写库 → 固定英文回执 → TTS → 结束 ——
    if intent == "create_ticket":
        priority = "P2" if "urgent" in message.lower() or "high" in message.lower() else "P3"
        tid, ts = create_ticket(name, company, email, message, priority)
        reply = f"A new support ticket has been created.\nTicket ID: {tid}\nCreated at: {ts}\nStatus: OPEN"
        audio_path = synthesize_speech(reply)
        yield reply, audio_path
        return

    # —— 查单：正则取号 → 查库 → 拼详情或错误 → TTS ——
    if intent == "check_ticket":
        match = re.search(r"(rt-\d+)", message.lower())
        if match:
            ticket_id = match.group(1).upper()
            data = get_ticket(ticket_id)
            if data:
                reply = (
                    f"Ticket {ticket_id} Details:\n"
                    f"Issue: {data['issue']}\n"
                    f"Status: {data['status']}\n"
                    f"Priority: {data['priority']}\n"
                    f"Created at: {data['created_at']}"
                )
            else:
                reply = f"No ticket found with ID {ticket_id}."
        else:
            reply = "Please provide a valid ticket ID."
        audio_path = synthesize_speech(reply)
        yield reply, audio_path
        return

    # —— 通用对话：system + 历史 + 当前 user；model 来自 UI 下拉 ——
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history_msgs + [{"role": "user", "content": message}]
    # stream=True：边生成边 yield，先不带音频（None）
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)

    full_reply = ""
    for chunk in stream:
        # delta.content 可能是 None，用 or "" 兜底
        delta = chunk.choices[0].delta.content or ""
        full_reply += delta
        # 流式阶段只更新文本，音频稍后一次合成
        yield full_reply, None  
    # 整段答完后再 TTS，最后一次 yield 带上音频路径
    audio_path = synthesize_speech(full_reply)
    yield full_reply, audio_path 


In [ ]:
# ========== 启动时建表：确保 tickets 表存在后再开 UI ==========
init_db()


In [ ]:
# ========== Gradio Blocks：表单信息 + 模型选择 + ChatInterface + 语音输出 ==========

# Soft 主题；浏览器标题保持英文原样
with gr.Blocks(title="Nova | Business AI Assistant", theme=gr.themes.Soft()) as demo:
    # 页面标题与说明（UI 文案字符串保持原样，避免改产品文案）
    gr.Markdown("## Nova | Reallytics.ai Customer Support & Sales Assistant")
    gr.Markdown(
        "Nova helps clients create or track support tickets, understand services, and explore automation options. "
        "Type your questions and Nova will respond in both text and voice."
    )

    # 第一行：姓名与公司（会作为建单字段传入 chat）
    with gr.Row():
        name = gr.Textbox(label="Your Name", placeholder="Liam")
        company = gr.Textbox(label="Company (optional)", placeholder="ABC Corp")
    # 邮箱单独一行
    email = gr.Textbox(label="Email", placeholder="you@example.com")

    # 模型下拉：选项与默认值（gpt-4o-mini 等）保持原样，会传给 chat 的 model 参数
    model = gr.Dropdown(["gpt-4o-mini", "gpt-4", "gpt-3.5-turbo"], value="gpt-4o-mini", label="Model")

    # 语音输出组件：autoplay=True 收到路径后自动播放
    audio_output = gr.Audio(label="Nova's Voice Reply", autoplay=True, interactive=False)

    # ChatInterface：fn=chat；type=messages；额外输入/输出接到上面的组件
    gr.ChatInterface(
        fn=chat,
        type="messages",
        additional_inputs=[model, name, company, email],
        additional_outputs=[audio_output],
        title="Chat with Nova",
        description="Ask about tickets, automation services, pricing, or integration and Nova will also speak her reply."
    )

# 脚本直接运行时启动 Gradio 服务（在笔记本里跑也会进入这里，取决于执行方式）
if __name__ == "__main__":
    demo.launch()
